In [ ]:
import pandas as pd
import joblib

# =====================================================
# SCRIPT DE PRÉDICTION DE LA CATÉGORIE DE PUISSANCE
# =====================================================

# Chargement du modèle déjà entraîné
modele = joblib.load("modele_puissance.pkl")

# Chargement du LabelEncoder pour convertir le résultat en texte
label_encoder = joblib.load("label_encoder_puissance.pkl")

# Chargement des colonnes utilisées pendant l'entraînement
colonnes_modele = joblib.load("colonnes_puissance.pkl")

# =====================================================
# ENTRÉE UTILISATEUR
# =====================================================

print("\nEntrez les caractéristiques de la borne :\n")

implantation_station = input(
    "Type d'implantation : "
)

nbre_pdc = int(
    input("Nombre de points de charge : ")
)

prise_type_ef = int(
    input("Prise EF disponible ? (1 = oui, 0 = non) : ")
)

prise_type_2 = int(
    input("Prise Type 2 disponible ? (1 = oui, 0 = non) : ")
)

prise_type_combo_ccs = int(
    input("Prise Combo CCS disponible ? (1 = oui, 0 = non) : ")
)

prise_type_chademo = int(
    input("Prise CHAdeMO disponible ? (1 = oui, 0 = non) : ")
)

prise_type_autre = int(
    input("Autre type de prise ? (1 = oui, 0 = non) : ")
)

gratuit = int(
    input("Borne gratuite ? (1 = oui, 0 = non) : ")
)

latitude = float(
    input("Latitude : ")
)

longitude = float(
    input("Longitude : ")
)

# =====================================================
# CRÉATION DU DATAFRAME DE LA NOUVELLE BORNE
# =====================================================

nouvelle_borne = pd.DataFrame([{
    "implantation_station": implantation_station,
    "nbre_pdc": nbre_pdc,
    "prise_type_ef": prise_type_ef,
    "prise_type_2": prise_type_2,
    "prise_type_combo_ccs": prise_type_combo_ccs,
    "prise_type_chademo": prise_type_chademo,
    "prise_type_autre": prise_type_autre,
    "gratuit": gratuit,
    "consolidated_latitude": latitude,
    "consolidated_longitude": longitude
}])

# =====================================================
# MÊME PRÉTRAITEMENT QUE PENDANT L'ENTRAÎNEMENT
# =====================================================

# Correction des accents pour correspondre à l'entraînement
nouvelle_borne["implantation_station"] = (
    nouvelle_borne["implantation_station"]
    .str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("utf-8")
)

# Transformation de l'implantation en colonnes numériques
nouvelle_borne = pd.get_dummies(
    nouvelle_borne,
    columns=["implantation_station"]
)

# Conversion des booléens éventuels
nouvelle_borne = nouvelle_borne.replace({
    "true": 1,
    "false": 0,
    "True": 1,
    "False": 0,
    True: 1,
    False: 0
})

# Conversion en numérique
for col in nouvelle_borne.columns:
    nouvelle_borne[col] = pd.to_numeric(
        nouvelle_borne[col],
        errors="coerce"
    )

# Ajout des colonnes absentes
# Exemple : si l'utilisateur entre "Voirie", les autres implantations n'existent pas dans la ligne
for col in colonnes_modele:
    if col not in nouvelle_borne.columns:
        nouvelle_borne[col] = 0

# Réorganisation des colonnes dans le même ordre que pendant l'entraînement
nouvelle_borne = nouvelle_borne[colonnes_modele]

# Conversion finale en float
nouvelle_borne = nouvelle_borne.astype(float)

# =====================================================
# PRÉDICTION
# =====================================================

prediction_num = modele.predict(nouvelle_borne)

prediction_texte = label_encoder.inverse_transform(
    prediction_num
)

print("\nRésultat :")
print(
    "Catégorie de puissance prédite :",
    prediction_texte[0]
)